<a href="https://colab.research.google.com/github/bsheese/387/blob/main/23_language/23_2_Pretrained_Transformers/23_2_1_Using_Pretrained_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Language: Part 4
## Using Pretrained Models

---

## What This Notebook Is About

23_0 and 23_1 opened up DistilBERT to see its tokens, embeddings and attention. This notebook uses pretrained language models the way most people meet them: as ready-made tools for a task, downloaded from Hugging Face and run with a few lines of code. Then it looks at what's inside one of those tools, what its documentation says, and where it goes wrong.

It is the text counterpart of 19_0, where we used a pretrained image model before knowing how it worked. The difference is that now you know what the parts are.

**What you will learn:**
1. How to run a ready-made sentiment model with a Hugging Face **pipeline**, and how well it does on our IMDB reviews
2. What a pipeline does: tokenizer, model, softmax, the steps from 23_0, 23_1 and 19_1_4
3. How to read a **model card**, and how to check a bias that the card reports
4. What DistilBERT was trained to do before anyone fine-tuned it: fill in missing words
5. **Zero-shot classification**: sorting texts into categories the model was never trained on, and how much the wording of the categories matters

The notebook downloads three models (about 1.2 GB together). It doesn't need a GPU; on Colab's CPU the slowest cell takes about a minute.

In [ ]:
import os
import time
import urllib.request

import numpy as np
import pandas as pd
import torch
from huggingface_hub import ModelCard
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

DATA_URL = 'https://raw.githubusercontent.com/bsheese/387/main/data/'

def fetch(name):
    # Download a file from the course's data folder, unless it is already here.
    # It saves to a temporary name first, so a download that fails partway doesn't leave a broken file behind.
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATA_URL + name, name + '.part')
        os.replace(name + '.part', name)
    return name

reviews = pd.read_csv(fetch('imdb_reviews.csv'))
test = reviews[reviews['split'] == 'test']
print(len(test), 'test reviews')

---

## Section 1: A Ready-Made Sentiment Model

A **pipeline** packages a pretrained model with everything needed to use it on raw text. You name the task and the model, and pass it strings. The model here, `distilbert-base-uncased-finetuned-sst-2-english`, is DistilBERT fine-tuned by Hugging Face to say whether a sentence is positive or negative. Each result is a label and a score, the probability the model gives that label.

In [ ]:
sentiment = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')

examples = [
    'One of the best films I have seen in years.',
    'The acting was fine, but the plot made no sense at all.',
    'Oh great, another two hours of my life I will never get back.',
    'I expected to hate it, and I did not.',
]
for text, result in zip(examples, sentiment(examples)):
    print(f"{result['label']:9} {result['score']:.3f}   {text}")

The first two are easy, and the model is almost certain about both. The last two are harder, and it gets both wrong. "Oh great, another two hours of my life I will never get back" is sarcastic, and the model calls it positive with a probability of 0.82, presumably because of "Oh great". "I expected to hate it, and I did not" is positive, but the model calls it negative with a probability of 0.997, the most confident of the four. The words "hate" and "not" usually go with negative reviews, and the model hasn't learned that here they cancel out. A score near 1 is no guarantee, the same lesson as the Pets model's confident mistakes in 19_0_2.

Now 100 of our IMDB test reviews. Reviews longer than 512 tokens, DistilBERT's limit (23_1_1), are cut off at 512.

In [ ]:
sample = test.sample(100, random_state=0)
start = time.time()
results = sentiment(list(sample['text']), truncation=True, max_length=512, batch_size=16)
predicted = np.array([r['label'] == 'POSITIVE' for r in results]).astype(int)
print(f'{len(sample)} reviews in {time.time() - start:.1f} s')
print(f"accuracy: {(predicted == sample['label'].values).mean():.2f}")

The model gets 91 of the 100 reviews right, in about two seconds on a laptop CPU. That's well above the 0.50 of always saying "positive" (every split of our reviews is half positive), and the model was never trained on IMDB reviews. 23_2_2 checks how it does on all 2,000 test reviews, next to a simple baseline and a model we fine-tune ourselves.

---

## Section 2: What the Pipeline Does

The pipeline is three steps you have already seen, joined together:

1. The **tokenizer** turns the text into token IDs, with `[CLS]` at the start and `[SEP]` at the end (23_0_1).
2. The **model** runs DistilBERT on the tokens (23_0_2, 23_1_1) and then a **classification head**, two small linear layers that turn the final vector of the `[CLS]` token into two scores, one per label.
3. **Softmax** turns the two scores into probabilities (19_1_4), and the pipeline reports the label with the higher one.

The next cell does the three steps by hand for one of the examples and compares the answer with the pipeline's.

In [ ]:
name = 'distilbert-base-uncased-finetuned-sst-2-english'
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name).eval()

text = examples[1]
inputs = tokenizer(text, return_tensors='pt')                       # step 1
print('tokens:', tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))
with torch.no_grad():
    scores = model(**inputs).logits[0]                              # step 2
probs = scores.softmax(dim=0)                                       # step 3
print('scores:       ', scores.numpy().round(3))
print('probabilities:', probs.numpy().round(3), ' labels:', model.config.id2label)
print('pipeline says:', sentiment(text)[0])
print('\nthe classification head:', model.pre_classifier, model.classifier, sep='\n')

The sentence becomes 16 tokens, with `[CLS]` first and `[SEP]` last. The model gives two scores, 4.13 for negative and −3.392 for positive, and softmax turns them into probabilities of 0.999 and 0.001. That is exactly the pipeline's answer: NEGATIVE with a score of 0.9994.

The classification head is two linear layers: the first keeps the 768 numbers of the `[CLS]` vector at 768, and the second turns them into the two scores. This is the same move as 19_0_1, where a new last layer was put on a pretrained network: the pretrained part produces a vector for the input, and a small trained layer turns that vector into an answer for the task. 23_2_2 adds a head like this one and trains it.

---

## Section 3: Reading the Model Card

Every model on Hugging Face has a **model card**: a page written by whoever shared the model, saying what it was trained on, what it's meant for, how well it scored and what its known problems are. It's the first thing to read before using a model for anything that matters. `ModelCard.load` downloads it; the cell below prints the parts we need.

In [ ]:
card = ModelCard.load(name)
print('trained on:', card.data.datasets)
for result in card.data.eval_results[:1]:
    print(f'reported {result.metric_type} on the {result.dataset_name} {result.dataset_split} split: {result.metric_value:.3f}')
text = card.text
start = text.index('## Risks, Limitations and Biases')
print()
print(text[start:start + 640])

The card says the model was fine-tuned on **SST-2**, the Stanford Sentiment Treebank: sentences and phrases from movie reviews on the website Rotten Tomatoes, each labeled positive or negative. It reports an accuracy of 0.911 on SST-2's validation sentences. That is a different set of reviews from ours, and SST-2's examples are single sentences, while ours are full reviews several paragraphs long, which may be part of why the model makes some of its mistakes on IMDB.

The section on risks reports something more serious: sentences of the form "This film was filmed in COUNTRY", which say nothing good or bad about the film, get very different probabilities depending on the country.

The card's claim is easy to test ourselves. Each sentence below differs only in the country, and none of them says anything good or bad about the film.

In [ ]:
countries = ['France', 'Italy', 'Japan', 'Canada', 'Mexico', 'Nigeria', 'India', 'Iran', 'Afghanistan']
results = sentiment([f'This film was filmed in {c}.' for c in countries])
for c, r in zip(countries, results):
    p_positive = r['score'] if r['label'] == 'POSITIVE' else 1 - r['score']
    print(f'{c:12} probability of positive {p_positive:.2f}')

The bias is there in our run, although the numbers aren't exactly the card's. France gets a probability of positive of 0.97 (the card says 0.89) and Afghanistan 0.16 (the card says 0.08). Most of the other countries are between 0.86 and 0.99, Nigeria is lower at 0.74, and Afghanistan stands far below the rest. (The small differences from the card probably come from a different version of the software or of the model's files since the card was written; the pattern is the same.)

The model learned from its training sentences which words tend to appear in positive and negative reviews, and nothing stopped it from attaching sentiment to the names of countries. We can't see exactly why from outside the model, but one possible reason is that in the text it learned from, Afghanistan was mentioned mostly in sentences about war, which are full of negative words. For a movie review that may not matter much, but the same model, or one made the same way, could be used to rate customer messages, job applications or social media posts, where a score that drops because a country is mentioned would treat people unfairly. The card's advice is to test for this kind of bias on your own use, as we just did, before relying on the model.

---

## Section 4: What DistilBERT Learned Before Fine-Tuning

The sentiment model was made from `distilbert-base-uncased`, the model from 23_0 and 23_1. That model was never trained to judge sentiment. It was **pretrained** on a large amount of ordinary text (English Wikipedia and a collection of books) with a fill-in-the-blank task called **masked language modeling**: hide some of the tokens in a passage and have the model predict them from the rest. No one has to label anything for this, because the hidden words themselves are the right answers, which is why it can be done on billions of words. (DistilBERT itself was trained to imitate a larger model, BERT, at this task; that is the "Distil" in its name.)

The `fill-mask` pipeline shows that pretraining at work: put `[MASK]` where a word should be, and it lists the model's top guesses with their probabilities.

In [ ]:
fill = pipeline('fill-mask', model='distilbert-base-uncased')
for sentence in ['This movie was absolutely [MASK].',
                 'The capital of France is [MASK].',
                 'The doctor said [MASK] would call back tomorrow.',
                 'The nurse said [MASK] would call back tomorrow.']:
    guesses = ', '.join(f"{g['token_str']} {g['score']:.2f}" for g in fill(sentence)[:5])
    print(f'{sentence:48} {guesses}')

The first guesses are sensible: "fantastic", "fabulous", "hilarious" and "incredible" all fit, and so does "terrible" (0.03), because pretraining only rewards words that fit the sentence, not words that are positive.

The second shows what pretraining doesn't do. The model's top guess for the capital of France is Marseille (0.14), and Paris is fourth (0.09). It has learned that the blank should be a French city, which is what fits the sentence's form, but it hasn't reliably stored the fact. Masked language modeling teaches what words are likely in a place, and that is not the same as what is true, a point 23_3_2 comes back to.

The last two show patterns from the training text. For the doctor, "he" (0.06) and "she" (0.05) are close, but for the nurse, "she" (0.12) is four times as likely as "he" (0.03). The model has picked up how often each pronoun appears near each job in its books and Wikipedia articles. Any model fine-tuned from it starts with these associations, which is how a bias like the one by country can end up in a sentiment model.

---

## Section 5: Zero-Shot Classification

Sometimes you need to sort texts into categories that no model was trained on, and you have no labeled examples to fine-tune with. **Zero-shot classification** handles this with a model trained on a different task, called **natural language inference** (NLI): given two sentences, decide whether the second one follows from the first. The pipeline turns each category into a sentence, by default "This example is about *category*.", asks the NLI model how strongly the text implies that sentence, and turns the answers for all the categories into probabilities that add up to 1.

The model here is a DeBERTa model (a relative of BERT) trained for exactly this use.

In [ ]:
zero_shot = pipeline('zero-shot-classification', model='MoritzLaurer/deberta-v3-base-zeroshot-v2.0')

texts = ['The battery died after two days and support never answered my emails.',
         'Great pasta, friendly staff, but we waited forty minutes for a table.',
         "The new update moved every button and now I can't find anything."]

def classify(labels):
    for text in texts:
        result = zero_shot(text, candidate_labels=labels)
        top = ', '.join(f'{label} {score:.2f}' for label, score in zip(result['labels'][:3], result['scores'][:3]))
        print(f'{text[:52]:54} {top}')

classify(['customer service', 'food', 'software', 'hardware', 'waiting time'])

With the first set of categories, the model gets one of three clearly right: the update complaint is about software (0.96). The restaurant review is split between waiting time (0.57) and food (0.41), which is fair, since it mentions both. But the battery complaint comes out as waiting time (0.64), when it is about a device that failed and about customer service. The probabilities add up to 1 across the categories, so a text about two things has to split its probability between them, and the model can still put most of it in the wrong place.

The categories are just words in a sentence, so their wording is part of the input. Here are the same three texts with the categories reworded to describe complaints more directly.

In [ ]:
classify(['bad customer service', 'bad food', 'a confusing app or program', 'a broken device', 'a long wait'])

With the reworded categories, the battery complaint goes to "bad customer service" (0.71), which is at least half right, though "a broken device" gets only 0.10. The restaurant review is now almost entirely "a long wait" (0.98), and the update complaint is "a confusing app or program" (0.88). Nothing about the texts or the model changed, only the words for the categories.

That makes zero-shot classification useful for a quick first sort when you have no labeled data, but its results can't be trusted without checking. The usual practice is to label a few dozen texts by hand, try several wordings of the categories on them, and keep the one that agrees best, which is choosing on a validation set, as in 19_1_1. (The pipeline also has a `multi_label=True` option, which scores each category on its own instead of making them share a total of 1, for texts that can be about several things.)

---

## Summary

- A Hugging Face **pipeline** runs a pretrained model on raw text: the tokenizer, the model with its **classification head**, and softmax, the steps from 23_0, 23_1 and 19_1_4.
- The ready-made sentiment model, fine-tuned on SST-2 sentences, got 91 of 100 IMDB test reviews right without ever being trained on IMDB, but it was fooled by sarcasm and by "I expected to hate it, and I did not".
- A **model card** says what a model was trained on, how it was evaluated and what its known problems are. This model's card reports a bias by country, and we could reproduce it.
- DistilBERT was **pretrained** with **masked language modeling**, predicting hidden words, which needs no labels. Its guesses show both what it learned about language and the patterns in its training text, such as which pronoun goes with "nurse".
- **Zero-shot classification** uses a model trained for **natural language inference** to sort texts into categories it was never trained on. It's useful when there are no labeled examples, but its answers depend on how the categories are worded.

## What's next

The sentiment model in this notebook was fine-tuned by someone else, on short sentences rather than full reviews. 23_2_2 does the fine-tuning ourselves: it starts from the same pretrained DistilBERT, adds a new classification head, trains it on our 2,000 IMDB training reviews with unit 21's training loop, and compares the result with a simple word-count baseline and with the ready-made model from this notebook.